In [35]:
import pandas as pd
import numpy as np

In [36]:
df_raw = pd.read_excel("student_performance_batchwise_2018_2025.xlsx")
df = df_raw.copy()
df = df.sort_values(['Student_ID', 'Semester'])

In [37]:
df.head()

,Student_ID,Student_Name,Batch,Semester,Year,Mid1,Mid2,Internals,EndTerm,Attendance,Total_Marks,SGPA,Performance_Type,Placement
0,S0001,Neha Pal,2018,1,2018,9.82,5.10,11.46,73.87,78.8,63.32,6.09,Low_Performer,Not Placed
1,S0001,Neha Pal,2018,2,2018,7.48,7.55,9.98,54.82,91.4,52.42,6.31,Low_Performer,Not Placed
2,S0001,Neha Pal,2018,3,2019,9.16,10.36,8.06,42.76,87.8,48.97,6.59,Low_Performer,Not Placed
3,S0001,Neha Pal,2018,4,2019,6.56,10.50,15.56,48.40,87.2,56.82,6.05,Low_Performer,Not Placed
4,S0001,Neha Pal,2018,5,2020,11.19,4.43,9.45,48.62,83.7,49.38,5.39,Low_Performer,Not Placed


In [38]:
# Recreate rolling dip flag
df_roll = df.copy()
df_roll['Rolling_Avg'] = df_roll.groupby('Student_ID')['SGPA'].transform(
    lambda x: x.expanding().mean()
)
df_roll['Rolling_STD'] = df_roll.groupby('Student_ID')['SGPA'].transform(
    lambda x: x.expanding().std()
).fillna(0)

In [39]:
df_roll['Is_Dip_Roll'] = df_roll['SGPA'] < (df_roll['Rolling_Avg'] - df_roll['Rolling_STD'])

In [40]:
print(df_roll.shape)  

(2752, 17)


In [41]:
df_roll.head()

,Student_ID,Student_Name,Batch,Semester,Year,Mid1,Mid2,Internals,EndTerm,Attendance,Total_Marks,SGPA,Performance_Type,Placement,Rolling_Avg,Rolling_STD,Is_Dip_Roll
0,S0001,Neha Pal,2018,1,2018,9.82,5.10,11.46,73.87,78.8,63.32,6.09,Low_Performer,Not Placed,6.090,0.000000,False
1,S0001,Neha Pal,2018,2,2018,7.48,7.55,9.98,54.82,91.4,52.42,6.31,Low_Performer,Not Placed,6.200,0.155563,False
2,S0001,Neha Pal,2018,3,2019,9.16,10.36,8.06,42.76,87.8,48.97,6.59,Low_Performer,Not Placed,6.330,0.250599,False
3,S0001,Neha Pal,2018,4,2019,6.56,10.50,15.56,48.40,87.2,56.82,6.05,Low_Performer,Not Placed,6.260,0.247925,False
4,S0001,Neha Pal,2018,5,2020,11.19,4.43,9.45,48.62,83.7,49.38,5.39,Low_Performer,Not Placed,6.086,0.444387,True


Collasping all Sems rows in single Row

In [42]:
def build_student_features(student_df):
    student_df = student_df.sort_values("Semester").copy()
    sgpa = student_df["SGPA"].values
    sems = student_df["Semester"].values

    
    expanding_mean = student_df['SGPA'].expanding().mean()
    expanding_std = student_df['SGPA'].expanding().std().fillna(0)
    is_dip = student_df['SGPA'] < (expanding_mean - expanding_std)

    
    # Slop for Trend
    if len(sgpa)>1:
        polyfit = np.polyfit(sems, sgpa, 1)
        slope = polyfit[0]
    else:
        slope = 0.0


    return pd.Series({
        'current_sgpa' : sgpa[-1],
        'sgpa_mean' : sgpa.mean(),
        "sgpa_trend" : round(slope, 4),
        'consistency_score' : sgpa.std(),
        'dip_count': int(is_dip.sum()),           # Total dip count
        'last_sem_dip': int(is_dip.iloc[-1]),      # Current dip status
        'attendance_mean': student_df['Attendance'].mean(),
        'internals_avg':     student_df['Internals'].mean()
        
    })
        
# Apply → 224 rows (one per student)
df_features = df_roll.groupby('Student_ID').apply(build_student_features).reset_index()


C:\Users\ani77\AppData\Local\Temp\ipykernel_8160\2021020316.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_features = df_roll.groupby('Student_ID').apply(build_student_features).reset_index()


In [43]:
print(df_features.shape)  # (224, 11)


(344, 9)


In [44]:
df_features.head()

,Student_ID,current_sgpa,sgpa_mean,sgpa_trend,consistency_score,dip_count,last_sem_dip,attendance_mean,internals_avg
0,S0001,5.03,5.89500,-0.1560,0.480833,3.0,1.0,83.4750,9.77250
1,S0002,7.15,7.59500,-0.0452,0.492874,0.0,0.0,72.7750,12.14000
2,S0003,7.90,7.07000,0.1838,0.482986,0.0,0.0,77.2500,11.51250
3,S0004,5.11,5.33875,-0.1763,0.571849,3.0,0.0,81.0125,9.85625
4,S0005,5.39,6.02625,-0.2235,0.553284,4.0,1.0,82.8375,10.76875


In [45]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [46]:
def build_training_features(student_df):
    features = build_student_features(student_df)
    # Add target label
    features["placement_ready"] = (
        student_df["Placement"].iloc[0]
    )
    return pd.Series(features)

df_features = (
    df_roll
    .groupby("Student_ID")
    .apply(build_training_features)
    .reset_index()
)


FEATURES = [
    "current_sgpa", "sgpa_mean", "sgpa_trend", "consistency_score", "dip_count", 'last_sem_dip',
    'attendance_mean', 'internals_avg'
]

X = df_features[FEATURES]

y = df_features["placement_ready"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

C:\Users\ani77\AppData\Local\Temp\ipykernel_8160\4110273485.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_training_features)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [47]:
print(classification_report(y_test, rf_model.predict(X_test),
      target_names=['Not Ready', 'Placement Ready']))

                 precision    recall  f1-score   support

      Not Ready       0.74      0.82      0.78        34
Placement Ready       0.81      0.71      0.76        35

       accuracy                           0.77        69
      macro avg       0.77      0.77      0.77        69
   weighted avg       0.77      0.77      0.77        69



In [48]:
from sklearn.metrics import accuracy_score
y_pred = rf_model.predict(X_test)
acc_RandomFor = accuracy_score(y_test, y_pred)
print(acc_RandomFor)

0.7681159420289855


In [49]:
importance = pd.Series(
    rf_model.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print(importance)

internals_avg        0.223005
current_sgpa         0.179413
sgpa_mean            0.168227
sgpa_trend           0.147328
consistency_score    0.129051
attendance_mean      0.097888
dip_count            0.043574
last_sem_dip         0.011513
dtype: float64


### Predicting a students v

### Only FOR Random Forest

In [50]:
def predict_student_readiness(student_df, rf_model, feature_list):
    # 1. Generate features automatically
    engineered_features = build_student_features(student_df)
    
    # 2. Convert to DataFrame and ensure correct column order
    input_df = pd.DataFrame([engineered_features])[feature_list]
    
    # 3. Get prediction and probability
    prediction = rf_model.predict(input_df)[0]
    probability = rf_model.predict_proba(input_df)[0][1]
    
    return {
        "is_ready": prediction,
        "readiness_score": round(probability * 100, 2),
        "total_dips_detected": engineered_features['dip_count'],
        "current_sem_is_dip": bool(engineered_features['last_sem_dip'])
    }

In [51]:
# Raw data: No 'Is_Dip_Roll' or 'Placement' columns needed
new_student = pd.DataFrame({
    "Semester": [1, 2, 3,4,5,6,7],
    "SGPA": [7.45, 8.12, 6.50, 7.12, 7.45, 7.6, 7.68], # Dropped from 8.12 to 6.50 (A Dip)
    "Attendance": [82, 85, 80, 70,72,22,15],
    "Internals": [21, 24, 23, 21, 24, 23, 20]
})

# Run prediction
result = predict_student_readiness(new_student, rf_model, FEATURES)
print(f"Readiness: {result['is_ready']}")
print(f"Probability: {result['readiness_score']}%")
print(f"Dips found: {result['total_dips_detected']}")

Readiness: Placed
Probability: 62.0%
Dips found: 1.0


## OTHER MODELS

### Logistic

In [52]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [53]:
y_pred = log_model.predict(X_test_scaled)
acc_logis = accuracy_score(y_test, y_pred)
print(acc_logis)

0.782608695652174


### XGBOOST

In [54]:
from xgboost import XGBClassifier
xg_model = XGBClassifier(n_estimators=100,max_depth=4,learning_rate=0.1,random_state=42,eval_metric='logloss'
)



y_xg = df_features["placement_ready"].map({
    "Not Placed": 0,
    "Placed": 1
})

X = df_features[FEATURES]
y = y_xg

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_xg,
    test_size=0.2,
    random_state=42,
    stratify=y_xg
)


xg_model.fit(X_train, y_train)
y_pred = xg_model.predict(X_test)


In [55]:
acc_XG = accuracy_score(y_test, y_pred)

acc_XG

0.7681159420289855

### SVM

In [56]:
from sklearn.svm import SVC


X = df_features[FEATURES]
# Binary target
y_svm = df_features["placement_ready"].map({
    "Not Placed": 0,
    "Placed": 1
})

X_train, X_test, y_train, y_test = train_test_split(
    X,y_svm,test_size=0.2,random_state=42,stratify=y_svm
)



In [57]:
svm_model = SVC(kernel='rbf',C=1.0,gamma='scale',random_state=42, probability=True)

svm_model.fit(X_train_scaled, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [58]:
y_pred = svm_model.predict(X_test_scaled)


acc_svm = accuracy_score(y_test, y_pred)

acc_svm

0.6956521739130435

In [59]:
print(acc_svm, acc_logis, acc_RandomFor, acc_XG)

0.6956521739130435 0.782608695652174 0.7681159420289855 0.7681159420289855


In [60]:
print(df_features["placement_ready"].value_counts())

placement_ready
Placed        173
Not Placed    171
Name: count, dtype: int64


In [61]:
df_features['Student_ID'].duplicated().sum()

np.int64(0)

## PREDICTION SYSTEM


In [62]:
def placement_predct(student_df, feature_list, model, scaler=None):

    engineered_features = build_student_features(student_df)

    input_df = pd.DataFrame([engineered_features])[feature_list]

    if scaler is not None:
        input_df = scaler.transform(input_df)

    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]

    return {"readiness_score" : round(probability * 100, 2),
            "total_dips_detected": engineered_features['dip_count'],
            "current_sem_is_dip": bool(engineered_features['last_sem_dip'])}

In [63]:
new_student = pd.DataFrame({
    "Semester": [1, 2, 3,4,5,6,7],
    "SGPA": [7.45, 8.12, 6.50, 6.50, 6.45, 5.6, 4.68], 
    "Attendance": [82, 85, 80, 70,72,22,15],
    "Internals": [21, 2, 23, 21, 10, 23, 20]
})




In [64]:
student_df = new_student
model = log_model
feature_list = FEATURES 


scaled_models = [log_model, svm_model]

if model in scaled_models:
    x = scaler
else:
    x = None

In [65]:
result = placement_predct(student_df, feature_list, model, scaler= x)
print(f"Probability: {result['readiness_score']}%")
print(f"Dips found: {result['total_dips_detected']}")


Probability: 94.47%
Dips found: 3.0


### Model's Prediction

In [66]:
models = {
    "Random Forest": rf_model,
    "Logistic Regression": log_model,
    "SVM": svm_model
}


for name, model in models.items():

    # Decide scaling automatically
    if model != rf_model:
        x = scaler
    else:
        x = None

    result = placement_predct(
        student_df,
        FEATURES,
        model,
        scaler=x
    )

    print(f"\n{name}")
    print(f"Probability: {result['readiness_score']}%")
    print(f"Dips Found: {result['total_dips_detected']}")


Random Forest
Probability: 57.0%
Dips Found: 3.0

Logistic Regression
Probability: 94.47%
Dips Found: 3.0

SVM
Probability: 44.74%
Dips Found: 3.0


#### Random Forest Feature Importance

In [67]:
importance = pd.Series(
    rf_model.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print(importance)

internals_avg        0.223005
current_sgpa         0.179413
sgpa_mean            0.168227
sgpa_trend           0.147328
consistency_score    0.129051
attendance_mean      0.097888
dip_count            0.043574
last_sem_dip         0.011513
dtype: float64


#### Logistic Regression Weightage

In [68]:
importance = pd.Series(
    log_model.coef_[0],
    index=FEATURES
).sort_values(ascending=False)

print(importance)

internals_avg        1.540331
current_sgpa         1.272334
last_sem_dip         0.096147
dip_count           -0.040609
attendance_mean     -0.102151
consistency_score   -0.135297
sgpa_trend          -0.158354
sgpa_mean           -1.232977
dtype: float64
